In [ ]:
import pandas as pd

# Load CSV
df = pd.read_csv("ams_features.csv")

# Keep only the columns you need
cols_to_keep = [
    "buurtnaam",
    "temp_mean",
    "buurt_area",
    "ndvi_mean",
    "water_prc",
    "road_prc"
]

df = df[cols_to_keep].copy()

# 3. Check the result
print(df.head())
print(df.info())

# 4. Save cleaned version
df.to_csv("heat_features_clean.csv", index=False)

: 

In [ ]:

import statsmodels.api as sm

# Load data
df = pd.read_csv("heat_features_clean.csv")

# Remove missing values
df = df.dropna()

# Remove rows with negative values
df = df[
    (df["temp_mean"] >= 0) &
    (df["ndvi_mean"] >= 0) &
    (df["water_prc"] >= 0) &
    (df["road_prc"] >= 0)
]

# Define target
y = df["temp_mean"]

# Define predictors
X = df[[
    "ndvi_mean",
    "water_prc",
    "road_prc"
]]

# Add intercept
X = sm.add_constant(X)

# Fit model
model = sm.OLS(y, X).fit()

# Results
print(model.summary())

In [ ]:
import geopandas as gpd

gdf = gpd.read_file("heat_ams.gpkg")


gdf["x"] = gdf.geometry.centroid.x
gdf["y"] = gdf.geometry.centroid.y

In [ ]:
import numpy as np

coords = list(zip(df["x"], df["y"]))

y = df["temp_mean"].values.reshape((-1,1))

X = df[[
    "ndvi_mean",
    "water_prc",
    "road_prc"
]].values

In [ ]:
from mgwr.sel_bw import Sel_BW

bw = Sel_BW(coords, y, X).search()

print(bw)

In [ ]:
from mgwr.gwr import GWR

gwr = GWR(coords, y, X, bw)

results = gwr.fit()

print(results.summary())

In [ ]:
gdf["ndvi_coef"] = results.params[:,1]

gdf.plot(
    column="ndvi_coef",
    legend=True
)